# Performance Optimization & Scalability
Demonstrates concrete techniques applied to the Gold layer to support
fast analytical queries, plus documented reasoning for how this design
would scale to much larger data volumes.

In [0]:
from pyspark.sql.functions import col

## 1. OPTIMIZE + Z-ORDER on fact_sales
fact_sales is the largest, most frequently joined/filtered table in the
star schema. Z-ordering physically co-locates rows with similar
customer_key/product_key values within the underlying Delta files,
so queries filtering or joining on these columns can skip
irrelevant data files instead of scanning everything.

In [0]:
spark.sql("OPTIMIZE workspace.gold.fact_sales ZORDER BY (customer_key, product_key)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

## 2. Table statistics for the query optimizer
ANALYZE collects column-level statistics (row counts, distinct values)
so Spark's query planner can make smarter decisions — e.g. choosing
a broadcast join when one side is small, instead of an expensive
shuffle join.

In [0]:
spark.sql("ANALYZE TABLE workspace.gold.fact_sales COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE workspace.gold.dim_customers COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE workspace.gold.dim_products COMPUTE STATISTICS FOR ALL COLUMNS")

DataFrame[]

## 3. Inspect the physical query plan
A typical analytical query (regional sales total) is run through
.explain() to inspect how Spark plans to execute it — confirming
join strategy and showing where filters are pushed down.

In [0]:
query_df=spark.sql("""
SELECT g.country, SUM(f.sales_amount) as total_sales
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.country
ORDER BY total_sales DESC
""")
query_df.explain(True)
display(query_df)

== Parsed Logical Plan ==
'Sort ['total_sales DESC NULLS LAST], true
+- 'Aggregate ['g.country], ['g.country, 'SUM('f.sales_amount) AS total_sales#38300]
   +- 'Join Inner, ('f.geography_key = 'g.geography_key)
      :- 'SubqueryAlias f
      :  +- 'UnresolvedRelation [workspace, gold, fact_sales], [], false
      +- 'SubqueryAlias g
         +- 'UnresolvedRelation [workspace, gold, dim_geography], [], false

== Analyzed Logical Plan ==
country: string, total_sales: double
Sort [total_sales#38300 DESC NULLS LAST], true
+- Aggregate [country#38326], [country#38326, sum(sales_amount#38325) AS total_sales#38300]
   +- Join Inner, (geography_key#38318 = geography_key#38327)
      :- SubqueryAlias f
      :  +- SubqueryAlias workspace.gold.fact_sales
      :     +- Relation workspace.gold.fact_sales[order_number#38315,customer_key#38316,product_key#38317,geography_key#38318,order_date_key#38319,order_date#38320,ship_date#38321,due_date#38322,quantity#38323,price#38324,sales_amount#38325] pa

country,total_sales
Australia,1.3380906E7
United States,1.3329428E7
United Kingdom,5181760.0
Germany,4358240.0
France,4085854.0
Canada,2798401.0
n/a,387332.0


## 4. Caching consideration (Serverless limitation)
On classic Databricks clusters, frequently-queried dimension tables
(e.g. dim_customers, dim_products) would typically be explicitly
cached in memory using .cache()/.persist() to speed up repeated
BI dashboard queries. Databricks Free Edition uses Serverless
compute, which manages caching automatically and does not support
explicit persist/cache calls (PERSIST TABLE is not supported on
serverless compute). Serverless compute has its own automatic
disk caching layer instead, so this is handled transparently
rather than manually.

## 5. Scalability design
This section documents design decisions that support scaling to
much larger data volumes, beyond what was actually tested here:

- **Delta Lake format**: supports file compaction (OPTIMIZE), data
  skipping via Z-ordering, and ACID transactions at any data volume —
  not just small datasets like this one.
- **Star schema shape**: fact_sales only stores surrogate key integers
  and measures, not repeated text/attribute data — keeping the fact
  table's row width small even as row count grows into millions.
- **Surrogate keys**: generated independently per dimension, so
  merging additional source systems later doesn't risk natural-key
  collisions.
- **Partitioning consideration**: for a truly large fact table, we
  would additionally partition fact_sales by order_date_key (or year)
  so queries filtering by date range only scan relevant partitions —
  not implemented here since the dataset is small, but a natural
  next step at scale.
- **Known current limitation**: surrogate key generation using
  row_number() without partitionBy forces a single-partition
  operation (seen as a Spark warning during dim_customers/dim_products
  builds). At large scale, this would be replaced with a distributed
  key generation strategy (e.g. monotonically_increasing_id() combined
  with partition offsets) to avoid a single-partition bottleneck.

## 6. Business Insight Queries
Demonstrates that the Gold star schema directly answers the three
insight categories required by the project: sales trends, customer
behavior, and regional performance — using only Gold tables.

##Sales trends over time

In [0]:
display(spark.sql("""
SELECT d.year, d.month, SUM(f.sales_amount) as total_sales, COUNT(*) as num_orders
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_date d ON f.order_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month
"""))

year,month,total_sales,num_orders
2010,12,44817.0,16
2011,1,479581.0,158
2011,2,476792.0,159
2011,3,497048.0,167
2011,4,515323.0,176
2011,5,574928.0,193
2011,6,756666.0,257
2011,7,613486.0,212
2011,8,631991.0,218
2011,9,615629.0,203


##Customer behavior

In [0]:
display(spark.sql("""
SELECT c.gender, c.marital_status, COUNT(DISTINCT f.customer_key) as customers, SUM(f.sales_amount) as total_sales, AVG(f.sales_amount) as avg_order_value
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_customers c ON f.customer_key = c.customer_key
WHERE c.is_current = true
GROUP BY c.gender, c.marital_status
"""))

gender,marital_status,customers,total_sales,avg_order_value
n/a,Single,7,14345.0,573.8
Female,Single,4385,1.1104934E7,521.0159519564605
n/a,Married,8,21977.0,813.9629629629629
Female,Married,4743,1.086912E7,463.3833560709413
Male,Single,4081,9695059.0,504.45179249700817
Male,Married,5260,1.1816486E7,458.41199518951004


## Regional performance

In [0]:
display(spark.sql("""
SELECT g.country, SUM(f.sales_amount) as total_sales, COUNT(*) as num_orders, AVG(f.sales_amount) as avg_order_value
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.country
ORDER BY total_sales DESC
"""))

country,total_sales,num_orders,avg_order_value
Australia,1.3380906E7,19672,680.2005896705978
United States,1.3329428E7,29904,445.7406367041198
United Kingdom,5181760.0,10701,484.23138024483694
Germany,4358240.0,8749,498.1415018859298
France,4085854.0,8545,478.15728496196607
Canada,2798401.0,10883,257.13507304971057
n/a,387332.0,1364,283.96774193548384


## top products 

In [0]:
display(spark.sql("""
SELECT p.category, p.subcategory, SUM(f.sales_amount) as total_sales
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_products p ON f.product_key = p.product_key
GROUP BY p.category, p.subcategory
ORDER BY total_sales DESC
"""))

category,subcategory,total_sales
Bikes,Road Bikes,2.0057472E7
Bikes,Mountain Bikes,1.7881481E7
Bikes,Touring Bikes,3844580.0
Accessories,Helmets,676305.0
Clothing,Jerseys,346884.0
Accessories,Tires and Tubes,244634.0
Clothing,Shorts,71330.0
Clothing,Gloves,68640.0
Clothing,Caps,59130.0
Accessories,Bottles and Cages,56993.0
